In [1]:
#!pip install crewai crewai_tools langchain langchain_community langchain_ollama streamlit duckduckgo-search

In [2]:
from crewai import LLM
from langchain_ollama.llms import OllamaLLM
from langchain_groq.chat_models import ChatGroq

llm = LLM(
    model="ollama/llama3.2:1b-instruct-q8_0",
    base_url="http://localhost:11434"
)

#llm = LLM(model="groq/openai/gpt-oss-20b")


In [3]:
from crewai.tools import tool
from langchain_community.tools import DuckDuckGoSearchResults 
import json


@tool
def search_web_tool(query: str):
    """
    Searches the web and returns results.
    """
    search_tool = DuckDuckGoSearchResults(num_results=5, verbose=True)
    return search_tool.run(query)


In [4]:
from crewai import Agent

# Agent Resercher
guide_expert= Agent( 
    role="City Local Guide Expert",
    goal="Provides information on things to do in the city based on the user's interests.",
    backstory="""A local expert with a passion for sharing the best experiences and hidden gems of their city.""",
    tools=[search_web_tool],
    verbose=True,
    max_iter=2,
    llm=llm,  #ChatOpenAI(temperature=0, model="gpt-4o-mini"),
    allow_delegation=False,
)

In [5]:
# Agent city expert
location_expert = Agent(
    role="Travel Trip Expert",
    goal="Adapt to the user destination vity language (French if city in French Country. Gather helpful information about to the city and city during travel.",
    backstory="""A seasoned traveler who has explored various destinations and knows the ins and outs of travel logistics.""",
    tools=[search_web_tool],
    verbose=True,
    max_iter=2,
    llm=llm,
    allow_delegation=False,
)


In [6]:
planner_expert = Agent(
    role="Travel Planning Expert",
    goal="Compiles all gathered information to provide a comprehensive travel plan.",
    backstory="""
    You are a professional guide with a passion for travel.
    An organizational wizard who can turn a list of possibilities into a seamless itinerary.
    """,
    tools=[search_web_tool],
    verbose=True,
    max_iter=2,
    llm=llm,
    allow_delegation=False,
)


In [7]:
from datetime import datetime
from crewai import Task

from_city = "India"
destination_city = "Rome"
date_from = "1st March 2025"
date_to = "7th March 2025"
interests = "sight seeing and good food"

location_task = Task(
    description=f"""
    In French : This task involves a comprehensive data collection process to provide the traveler with essential information about their destination. It includes researching and compiling details on various accommodations, ranging from budget-friendly hostels to luxury hotels, as well as estimating the cost of living in the area. The task also covers transportation options, visa requirements, and any travel advisories that may be relevant.
    consider also the weather conditions forcast on the travel dates. and all the events that may be relevant to the traveler during the trip period.
    
    Traveling from : {from_city}
    Destination city : {destination_city}
    Arrival Date : {date_from}
    Departure Date : {date_to}

    Follow this rules : 
    1. if the {destination_city} is in a French country : Respond in FRENCH.
    """,
    expected_output=f"""
    if the {destination_city} is in a French country : Respond in FRENCH.
    In markdown format : A detailed markdown report that includes a curated list of recommended places to stay, a breakdown of daily living expenses, and practical travel tips to ensure a smooth journey.
    """,
    agent=location_expert,
    output_file='agentic-ai/crewai/city_report.md',
)





In [8]:
guide_task = Task(
    description=f"""
    if the {destination_city} is in a French country : Respond in FRENCH.
    Tailored to the traveler's personal {interests}, this task focuses on creating an engaging and informative guide to the city's attractions. It involves identifying cultural landmarks, historical spots, entertainment venues, dining experiences, and outdoor activities that align with the user's preferences such {interests}. The guide also highlights seasonal events and festivals that might be of interest during the traveler's visit.
    Destination city : {destination_city}
    interests : {interests}
    Arrival Date : {date_from}
    Departure Date : {date_to}

    Follow this rules : 
    1. if the {destination_city} is in a French country : Respond in FRENCH.
    """,
    expected_output=f"""
    An interactive markdown report that presents a personalized itinerary of activities and attractions, complete with descriptions, locations, and any necessary reservations or tickets.
    """,

    agent=guide_expert,
    output_file='agentic-ai/crewai/guide_report.md',
)


In [9]:
planner_task = Task(
    description=f"""
    This task synthesizes all collected information into a detaileds introduction to the city (description of city and presentation, in 3 pragraphes) cohesive and practical travel plan. and takes into account the traveler's schedule, preferences, and budget to draft a day-by-day itinerary. The planner also provides insights into the city's layout and transportation system to facilitate easy navigation.
    Destination city : {destination_city}
    interests : {interests}
    Arrival Date : {date_from}
    Departure Date : {date_to}

    Follow this rules : 
    1. if the {destination_city} is in a French country : Respond in FRENCH.
    """,
    expected_output="""
    if the {destination_city} is in a French country : Respond in FRENCH.
    A rich markdown document with emojis on each title and subtitle, that :
    In markdown format : 
    # Welcome to {destination_city} :
    A 4 paragraphes markdown formated including :
    - a curated articles of presentation of the city, 
    - a breakdown of daily living expenses, and spots to visit.
    # Here's your Travel Plan to {destination_city} :
    Outlines a daily detailed travel plan list with time allocations and details for each activity, along with an overview of the city's highlights based on the guide's recommendations
    """,
    context=[location_task, guide_task],
    #context=context,
    agent=planner_expert,
    output_file='agentic-ai/crewai/travel_plan.md',
)


In [10]:
# Task: Location
def location_task(agent, from_city, destination_city, date_from, date_to):
    return Task(
        description=f"""
        In French : This task involves a comprehensive data collection process to provide the traveler with essential information about their destination. It includes researching and compiling details on various accommodations, ranging from budget-friendly hostels to luxury hotels, as well as estimating the cost of living in the area. The task also covers transportation options, visa requirements, and any travel advisories that may be relevant.
        consider also the weather conditions forcast on the travel dates. and all the events that may be relevant to the traveler during the trip period.
        
        Traveling from : {from_city}
        Destination city : {destination_city}
        Arrival Date : {date_from}
        Departure Date : {date_to}

        Follow this rules : 
        1. if the {destination_city} is in a French country : Respond in FRENCH.
        """,
        expected_output=f"""
        if the {destination_city} is in a French country : Respond in FRENCH.
        In markdown format : A detailed markdown report that includes a curated list of recommended places to stay, a breakdown of daily living expenses, and practical travel tips to ensure a smooth journey.
        """,
        agent=agent,
        output_file='agentic-ai/crewai/city_report.md',
    )

# Task: Location
def guide_task(agent, destination_city, interests, date_from, date_to):    
    return Task(
        description=f"""
        if the {destination_city} is in a French country : Respond in FRENCH.
        Tailored to the traveler's personal {interests}, this task focuses on creating an engaging and informative guide to the city's attractions. It involves identifying cultural landmarks, historical spots, entertainment venues, dining experiences, and outdoor activities that align with the user's preferences such {interests}. The guide also highlights seasonal events and festivals that might be of interest during the traveler's visit.
        Destination city : {destination_city}
        interests : {interests}
        Arrival Date : {date_from}
        Departure Date : {date_to}

        Follow this rules : 
        1. if the {destination_city} is in a French country : Respond in FRENCH.
        """,
        expected_output=f"""
        An interactive markdown report that presents a personalized itinerary of activities and attractions, complete with descriptions, locations, and any necessary reservations or tickets.
        """,

        agent=agent,
        output_file='agentic-ai/crewai/guide_report.md',
    )


# Task: Planner
def planner_task(context, agent, destination_city, interests, date_from, date_to):
    return Task(
        description=f"""
        This task synthesizes all collected information into a detaileds introduction to the city (description of city and presentation, in 3 pragraphes) cohesive and practical travel plan. and takes into account the traveler's schedule, preferences, and budget to draft a day-by-day itinerary. The planner also provides insights into the city's layout and transportation system to facilitate easy navigation.
        Destination city : {destination_city}
        interests : {interests}
        Arrival Date : {date_from}
        Departure Date : {date_to}

        Follow this rules : 
        1. if the {destination_city} is in a French country : Respond in FRENCH.
        """,
        expected_output="""
        if the {destination_city} is in a French country : Respond in FRENCH.
        A rich markdown document with emojis on each title and subtitle, that :
        In markdown format : 
        # Welcome to {destination_city} :
        A 4 paragraphes markdown formated including :
        - a curated articles of presentation of the city, 
        - a breakdown of daily living expenses, and spots to visit.
        # Here's your Travel Plan to {destination_city} :
        Outlines a daily detailed travel plan list with time allocations and details for each activity, along with an overview of the city's highlights based on the guide's recommendations
        """,
        #context=[location_task, guide_task],
        context=context,
        agent=agent,
        output_file='agentic-ai/crewai/travel_plan.md',
    )


In [11]:

location_task = location_task(
  location_expert,
  from_city,
  destination_city,
  date_from,
  date_to
)

guide_task = guide_task(
  guide_expert,
  destination_city,
  interests,
  date_from,
  date_to
)

planner_task = planner_task(
  [location_task, guide_task],
  planner_expert,
  destination_city,
  interests,
  date_from,
  date_to,
)



In [12]:
from crewai import Crew, Process
crew = Crew(
    agents=[location_expert, guide_expert, planner_expert],
    tasks=[location_task, guide_task, planner_task],
    process=Process.sequential,
    full_output=True,
    share_crew=False,
    verbose=True
)

result = crew.kickoff()

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: ae4ad7c8-fef2-40d0-9a21-3628993c67b2                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│          In French : This task involves a comprehensive data collection process to provide the traveler with    │
│  essential information about their destination. It includes researching and compiling details on various        │
│  accommodations, ranging from budget-friendly hostels to luxury hotels, as well as estimating the cost of       │
│  living in the area. The task also covers transportation options, visa requirements, and any travel advisories  │
│  that may be relevant.                                                                                          │
│          consider also the weather conditions forcast on the travel dates. and all the events that may be       │
│  relevant to the traveler during the trip period.                                                               │
│                                                                                                                 │
│          Traveling from : India                                                                                 │
│          Destination city : Rome                                                                                │
│          Arrival Date : 1st March 2025                                                                          │
│          Departure Date : 7th March 2025                                                                        │
│                                                                                                                 │
│          Follow this rules :                                                                                    │
│          1. if the Rome is in a French country : Respond in FRENCH.                                             │
│                                                                                                                 │
│  ID: 310f28e0-9405-451f-94d3-4ee8f8de6d4b                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Travel Trip Expert                                                                                      │
│                                                                                                                 │
│  Task:                                                                                                          │
│          In French : This task involves a comprehensive data collection process to provide the traveler with    │
│  essential information about their destination. It includes researching and compiling details on various        │
│  accommodations, ranging from budget-friendly hostels to luxury hotels, as well as estimating the cost of       │
│  living in the area. The task also covers transportation options, visa requirements, and any travel advisories  │
│  that may be relevant.                                                                                          │
│          consider also the weather conditions forcast on the travel dates. and all the events that may be       │
│  relevant to the traveler during the trip period.                                                               │
│                                                                                                                 │
│          Traveling from : India                                                                                 │
│          Destination city : Rome                                                                                │
│          Arrival Date : 1st March 2025                                                                          │
│          Departure Date : 7th March 2025                                                                        │
│                                                                                                                 │
│          Follow this rules :                                                                                    │
│          1. if the Rome is in a French country : Respond in FRENCH.                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Travel Trip Expert                                                                                      │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  {function search_web_tool                                                                                      │
│      {object search_results                                                                                     │
│  [{"query":{"type":"string"},"search_term1":"place_name","location":[{"type":"string","value":"Italy"},         │
│  {"type":"integer","value":92159},{"type":"null","value":false}]}}                                              │
│  }                                                                                                              │
│                                                                                                                 │
│  {function travel_log                                                                                           │
│      {object travel_info                                                                                        │
│          "source": ["India"],                                                                                   │
│          "destination": ["Rome"],                                                                               │
│          "arrival_date": {"date_type": "datetime", "date_value": "2025-03-01"},                                 │
│          "departure_date": {"date_type": "datetime", "date_value": "2025-03-07"}                                │
│      }                                                                                                          │
│  }                                                                                                              │
│                                                                                                                 │
│  {function weather forecast                                                                                     │
│      {object weather_info                                                                                       │
│          "temperature_max": {"value": 26, "unit": "°C"}                                                         │
│          "weather_description": {"type": "string", "content": "partly cloudy"}                                  │
│              [object search_web_tool]                                                                           │
│  }                                                                                                              │
│                                                                                                                 │
│  if "Rome" in ["Paris"] {                                                                                       │
│      return "Bonjour! Les destinations recommandées pour votre séjour à Rome sont : \n"                         │
│                                                                                                                 │
│      result = { object: [search_web_tool, [search_results] ], data: [] }                                        │
│                                                                                                                 │
│      for result1 in result["data"]:                                                                             │
│          data2 = [object]: []                                                                                   │
│          if "description" in search_result1:           

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│          In French : This task involves a comprehensive data collection process to provide the traveler with    │
│  essential information about their destination. It includes researching and compiling details on various        │
│  accommodations, ranging from budget-friendly hostels to luxury hotels, as well as estimating the cost of       │
│  living in the area. The task also covers transportation options, visa requirements, and any travel advisories  │
│  that may be relevant.                                                                                          │
│          consider also the weather conditions forcast on the travel dates. and all the events that may be       │
│  relevant to the traveler during the trip period.                                                               │
│                                                                                                                 │
│          Traveling from : India                                                                                 │
│          Destination city : Rome                                                                                │
│          Arrival Date : 1st March 2025                                                                          │
│          Departure Date : 7th March 2025                                                                        │
│                                                                                                                 │
│          Follow this rules :                                                                                    │
│          1. if the Rome is in a French country : Respond in FRENCH.                                             │
│                                                                                                                 │
│  Agent: Travel Trip Expert                                                                                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Travel Planning Expert                                                                                  │
│                                                                                                                 │
│  Task:                                                                                                          │
│          This task synthesizes all collected information into a detaileds introduction to the city              │
│  (description of city and presentation, in 3 pragraphes) cohesive and practical travel plan. and takes into     │
│  account the traveler's schedule, preferences, and budget to draft a day-by-day itinerary. The planner also     │
│  provides insights into the city's layout and transportation system to facilitate easy navigation.              │
│          Destination city : Rome                                                                                │
│          interests : sight seeing and good food                                                                 │
│          Arrival Date : 1st March 2025                                                                          │
│          Departure Date : 7th March 2025                                                                        │
│                                                                                                                 │
│          Follow this rules :                                                                                    │
│          1. if the Rome is in a French country : Respond in FRENCH.                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Travel Planning Expert                                                                                  │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  I'll provide you with a JSON response that meets the specified requirements. Before I start, please note that  │
│  I need to update your code to ensure it correctly handles different countries and their respective layout.     │
│                                                                                                                 │
│  Here is the final answer:                                                                                      │
│                                                                                                                 │
│  ```json                                                                                                        │
│  {                                                                                                              │
│      "name": "{function:search_web_tool}",                                                                      │
│      "parameters": {                                                                                            │
│          "query": "",                                                                                           │
│          "search_term1":"destination_country","location":[{"type":"string","value":"",                          │
│             {"type":"integer","value"}},                                                                        │
│            {"type":"string","value":"","null":["",2000,2005"]],                                                 │
│            {"type":"vector","dimension":2,"values":[12000000, 12250000]}                                        │
│        }                                                                                                        │
│      }                                                                                                          │
│  }                                                                                                              │
│  {"_meta_:travel_log, _format": "[name:Travel Log]", "data": ["Rome is not located in a French country."]}      │
│  {"_meta_:transportation", _format": "<|reserved_special_token_1|- Transportation plan for {destination_city}   │
│  : Rome\nTraveling from city {origin} to Rome by plane:\n  * Departure from Paris, France: Flight duration 7    │
│  hours and estimated cost €100 - €300. Add to your travel itinerary in the given time.\n Traveling by train or  │
│  car from Paris, France to Rome is also an option:\n  * Take train L5 from Paris to Termini station             │
│  (approximate duration 2 hours, approximate journey duration 7-8 hours)\n  * Drive from Paris to Rome via       │
│  highways A41 and A1. Estimated cost: €50 approx., total estimated travel duration approximately 16-18          │
│  hours"},                                                                                                       │
│      "data": [                                                                                                  │
│                                                                                                                 │
│          {"object": "{function:travel_log}", "data": ["Travel Log"]},                                           │
│                                                                                                                 │
│          [                                             

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│          This task synthesizes all collected information into a detaileds introduction to the city              │
│  (description of city and presentation, in 3 pragraphes) cohesive and practical travel plan. and takes into     │
│  account the traveler's schedule, preferences, and budget to draft a day-by-day itinerary. The planner also     │
│  provides insights into the city's layout and transportation system to facilitate easy navigation.              │
│          Destination city : Rome                                                                                │
│          interests : sight seeing and good food                                                                 │
│          Arrival Date : 1st March 2025                                                                          │
│          Departure Date : 7th March 2025                                                                        │
│                                                                                                                 │
│          Follow this rules :                                                                                    │
│          1. if the Rome is in a French country : Respond in FRENCH.                                             │
│                                                                                                                 │
│  Agent: Travel Planning Expert                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: ae4ad7c8-fef2-40d0-9a21-3628993c67b2                                                                       │
│  Final Output: I'll provide you with a JSON response that meets the specified requirements. Before I start,     │
│  please note that I need to update your code to ensure it correctly handles different countries and their       │
│  respective layout.                                                                                             │
│                                                                                                                 │
│  Here is the final answer:                                                                                      │
│                                                                                                                 │
│  ```json                                                                                                        │
│  {                                                                                                              │
│      "name": "{function:search_web_tool}",                                                                      │
│      "parameters": {                                                                                            │
│          "query": "",                                                                                           │
│          "search_term1":"destination_country","location":[{"type":"string","value":"",                          │
│             {"type":"integer","value"}},                                                                        │
│            {"type":"string","value":"","null":["",2000,2005"]],                                                 │
│            {"type":"vector","dimension":2,"values":[12000000, 12250000]}                                        │
│        }                                                                                                        │
│      }                                                                                                          │
│  }                                                                                                              │
│  {"_meta_:travel_log, _format": "[name:Travel Log]", "data": ["Rome is not located in a French country."]}      │
│  {"_meta_:transportation", _format": "<|reserved_special_token_1|- Transportation plan for {destination_city}   │
│  : Rome\nTraveling from city {origin} to Rome by plane:\n  * Departure from Paris, France: Flight duration 7    │
│  hours and estimated cost €100 - €300. Add to your travel itinerary in the given time.\n Traveling by train or  │
│  car from Paris, France to Rome is also an option:\n  * Take train L5 from Paris to Termini station             │
│  (approximate duration 2 hours, approximate journey duration 7-8 hours)\n  * Drive from Paris to Rome via       │
│  highways A41 and A1. Estimated cost: €50 approx., total estimated travel duration approximately 16-18          │
│  hours"},                                                                                                       │
│      "data": [                                                                                                  │
│                                                                                                                 │
│          {"object": "{function:travel_log}", "data": ["Travel Log"]},                                           │
│                                                       

╭────────────────────────── Trace Batch Finalization ──────────────────────────╮
│ ✅ Trace batch finalized with session ID:                                    │
│ 26800519-21ca-44cb-b518-b0437678ae18                                         │
│                                                                              │
│ 🔗 View here:                                                                │
│ https://app.crewai.com/crewai_plus/ephemeral_trace_batches/26800519-21ca-44c │
│ b-b518-b0437678ae18?access_code=TRACE-f03127399f                             │
│ 🔑 Access Code: TRACE-f03127399f                                             │
╰──────────────────────────────────────────────────────────────────────────────╯
